# 공통 설정: RAGaaS 요청과 모의 응답

원본: [`ragaas.py`](ragaas.py)

사내 RAGaaS 요청과 사외에서 API 흐름을 연습할 수 있는 모의 응답을 한 함수에서 선택한다.

## 학습 목표

- `RAG_PROVIDER`에 따른 분기를 찾는다.
- 실제 요청에 필요한 설정 이름과 요청 헤더를 구분한다. 비밀값은 출력하지 않는다.
- `MockResponse`의 응답이 실제 문서 내용이 아니라는 표시를 확인한다.

## 실행 전 확인

아래 원본 코드 셀은 함수와 클래스를 정의하며 요청을 보내지 않는다. `post_ragaas()`를 호출할 때 기본 `samsung` 경로는 사내망과 인증이 필요하다. 사외에서는 `RAG_PROVIDER=mock`을 설정한 후 실습한다.

In [ ]:
"""사내 RAGaaS 요청과 사외 실습용 모의 응답."""

import json
import os

from dotenv import load_dotenv

load_dotenv()

_ENDPOINTS = {"insert-doc", "retrieve-rrf", "delete-doc"}


class MockResponse:
    """requests.Response에서 실습 코드가 사용하는 부분만 제공한다."""

    status_code = 200

    def __init__(self, body: dict):
        self._body = body
        self.text = json.dumps(body, ensure_ascii=False)

    def json(self) -> dict:
        return self._body

    def raise_for_status(self) -> None:
        pass


def post_ragaas(endpoint: str, payload: dict, *, api_key: str | None = None):
    """RAG_PROVIDER=mock이면 네트워크 없이 같은 응답 형태를 돌려준다."""
    if endpoint not in _ENDPOINTS:
        raise ValueError(f"지원하지 않는 RAGaaS 엔드포인트: {endpoint}")

    provider = os.getenv("RAG_PROVIDER", "samsung").strip().lower()
    if provider == "mock":
        if endpoint == "retrieve-rrf":
            question = payload.get("query_text", "")
            return MockResponse({
                "hits": {
                    "hits": [{
                        "_source": {
                            "merge_title_content": (
                                f"[MOCK RAGaaS] 질문: {question}. "
                                "이 문장은 API 요청·응답 흐름을 연습하기 위한 모의 자료이며 "
                                "실제 사내 문서나 보험 약관의 내용이 아닙니다."
                            )
                        }
                    }]
                }
            })
        return MockResponse({"status": "mock", "endpoint": endpoint})

    if provider != "samsung":
        raise ValueError(f"지원하지 않는 RAG_PROVIDER: {provider!r}. 'mock' 또는 'samsung'을 사용하세요.")

    required = ("search_url", "rag_credential_key")
    missing = [name for name in required if not os.getenv(name)]
    api_key = api_key or os.getenv("rag_api_key")
    if not api_key:
        missing.append("rag_api_key")
    if missing:
        raise RuntimeError(f"사내 RAGaaS 설정이 부족합니다: {missing}")

    import requests

    response = requests.post(
        f'{os.environ["search_url"].rstrip("/")}/dsllmrag/elastic/v2/{endpoint}',
        json=payload,
        headers={
            "Content-Type": "application/json",
            "x-dep-ticket": os.environ["rag_credential_key"],
            "api-key": api_key,
        },
        timeout=30,
    )
    response.raise_for_status()
    return response


## 안전한 로컬 확인

아래 셀은 네트워크를 사용하지 않고 모의 응답의 구조만 살핀다.

In [ ]:
sample = MockResponse({"hits": {"hits": []}})
assert sample.status_code == 200
assert sample.json()["hits"]["hits"] == []
print("모의 응답 구조 확인 완료")


## CodeMate와 함께하기

> `src/config/ragaas.py`의 `mock`·`samsung` 분기를 읽고, 두 경로가 각각 어떤 입력을 검사하고 어떤 응답을 반환하는지 표로 정리해 줘. 실제 인증 정보는 출력하지 마. 이어서 잘못된 endpoint와 지원하지 않는 provider에서 드러나는 오류를 설명해 줘.

## 결과 검토

1. 모의 응답과 실제 사내 문서를 혼동하지 않으려면 화면에 어떤 표시가 필요한가?
2. `rag_api_key`가 없을 때 어느 지점에서 실패하는가?
3. `retrieve-rrf` 이외 endpoint를 요청하면 모의 응답 형태는 어떻게 다른가?